# 19 — Robustness Benchmarking & Red-Teaming Toolkits — Lab

## Description

This lab exercises automated vulnerability auditing using IBM ART. You load a pretrained CIFAR-10 classifier, run three standardized attacks (FGSM, PGD, C&W), record per-attack metrics (fooling rate and perturbation magnitudes), and produce a structured vulnerability report. You then situate the model on the RobustBench leaderboard to contextualize its robustness.

## Objectives

- Student can use IBM ART to run a multi-attack robustness audit and produce a structured vulnerability report with per-attack metrics.
- Student can explain what AutoAttack is, why it uses four complementary attacks, and why it is preferred over single-attack evaluation for leaderboard comparisons.
- Student can situate a model on the RobustBench leaderboard by its AutoAttack accuracy.

## Prerequisites

You should have completed **Lesson 06** (AI Development Lifecycle and MLOps) to understand model versioning and experiment tracking. You should have studied m13 (Physical-World Attacks and Universal Perturbations) for familiarity with perturbation budgets and attack evaluation. You must have completed the lecture content for this module (**Lesson 19**) covering **RobustBench**, **AutoAttack**, ART architecture, and red-teaming phases.

## Required Software/Packages

- PyTorch >= 1.9.0
- adversarial-robustness-toolbox >= 1.10.0
- numpy >= 1.19.0
- pandas >= 1.1.0
- torchvision >= 0.10.0 (for CIFAR-10 dataset)

## Environment/Setup Instructions

Run the cell below to import all required libraries and verify that ART and PyTorch are installed correctly.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, CarliniL2Method
import numpy as np
import pandas as pd

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("All imports successful. Ready to begin.")

## Background/Context

In the lecture, you learned that AutoAttack is an ensemble of four complementary attacks, and that RobustBench uses AutoAttack to provide standardized robustness evaluation. In practice, organizations run multi-attack audits using tools like IBM ART to assess model vulnerability before deployment. This lab replicates the core audit workflow: you load a model, run three key attacks from the AutoAttack ensemble (APGD-CE approximated by PGD, APGD-DLR approximated by C&W, and a fast boundary attack approximated by C&W), and produce a vulnerability report that summarizes the findings. The report shows per-attack fooling rate (percentage of examples fooled) and perturbation magnitudes, allowing you to prioritize defenses based on which attack is most effective.

## Exercise Instructions

1. Define a simple CNN model for CIFAR-10 classification with two convolutional layers and a fully connected head. (2 min)
2. Load the CIFAR-10 test dataset, selecting the first 200 examples for this audit. (2 min)
3. Create a PyTorchClassifier wrapper by passing your model and preprocessing function to ART. (3 min)
4. Instantiate three attacks via ART: FGSM (epsilon=8/255), PGD-20 (epsilon=8/255, alpha=2/255, max_iter=20), and C&W L2 (kappa=0, max_iter=50). (5 min)
5. Run each attack on the test batch and record: fooling rate (%), mean L2 perturbation magnitude, and mean L-infinity perturbation magnitude. (10 min)
6. Create a structured audit report as a pandas DataFrame with one row per attack and the three metrics as columns. (5 min)
7. Compare your model's implied AutoAttack accuracy to the RobustBench leaderboard snapshot provided. (3 min)

### Define model and create ART classifier

**Options**

- Train your own model
- Load pre-trained model

In [ ]:
# If you want to load models from RobustBench, you need to install it.
import warnings
warnings.filterwarnings('ignore')

try:
    import robustbench
except ImportError:
    !!pip install -q git+https://github.com/RobustBench/robustbench.git@2d630bc9e8d1cf50d46a4dda65fd36850e3ef769
    import robustbench
    print("RobustBench installed successfully.")

Let's install version [AutoAttack](https://arxiv.org/abs/2003.01690) from 
ICML 2020 with 2/4 attacks (only APGD-CE and APGD-DLR)

In [ ]:
# now let's get the ## AutoAttack evaluation
!pip install -q git+https://github.com/fra31/auto-attack

### Load the data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

"""
# Load CIFAR-10 test data
transform = transforms.Compose([transforms.ToTensor()])
cifar10_test = datasets.CIFAR10(root='/tmp/cifar10', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(cifar10_test, batch_size=200, shuffle=False)

x_test, y_test = next(iter(test_loader))
x_test = x_test.numpy()
y_test = y_test.numpy()
print(f"Test batch shape: {x_test.shape}, Test labels shape: {y_test.shape}")

"""

from robustbench.data import load_cifar10
x_test, y_test = load_cifar10(n_examples=100)


In [ ]:
# Option 1: Define your own model

"""
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x


model = SimpleCNN().to(device)
print(f"Model initialized on {device}")

"""

In [ ]:
# Option 2: Load a pre-trained model 
# Example using torchvision or RobustBench

from robustbench.utils import load_model
from robustbench.utils import list_available_models
list_available_models(dataset="cifar10", threat_model="Linf")

In [ ]:
# load your choice ofpre-trained model from RobustBench
model = load_model(
    model_name="Carmon2019Unlabeled",  # model from the CIFAR-10 leaderboard
    dataset="cifar10",
    threat_model="Linf"   # "Linf", "L2", or "corruptions"
)
model.eval()  # don't forget eval mode
model = model.to(device)  # move model to the appropriate device

In [ ]:
# Create ART PyTorchClassifier
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

classifier = PyTorchClassifier(
    model=model,
    loss=loss_fn,
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=10,
    device_type="gpu" if torch.cuda.is_available() else "cpu"
)

predictions = classifier.predict(x_test)
clean_accuracy = np.mean(np.argmax(predictions, axis=1) == y_test)
print(f"Clean accuracy on test examples: {clean_accuracy:.1%}")

In [ ]:
# Steps 4-6: Run attacks using ART and create audit report
epsilon = 8/255
attack_results = {}

def evaluate_attack(classifier, x_adv, x_clean, y_true, attack_name, results_dict):
    """
    Compute fooling rate and perturbation norms for an adversarial attack,
    store them in results_dict, and print a summary.

    Returns the metrics dict.
    """
    pred_adv = classifier.predict(x_adv)
    fooling_rate = np.mean(np.argmax(pred_adv, axis=1) != y_true) * 100

    pert = x_adv - x_clean
    mean_l2 = np.mean(np.linalg.norm(pert.reshape(pert.shape[0], -1), axis=1))
    mean_linf = np.mean(np.abs(pert).max(axis=(1, 2, 3)))

    metrics = {'fooling_rate': fooling_rate, 'mean_l2': mean_l2, 'mean_linf': mean_linf}
    results_dict[attack_name] = metrics

    print(f"{attack_name} - Fooling rate: {fooling_rate:.1f}%, "
          f"Mean L2: {mean_l2:.4f}, Mean Linf: {mean_linf:.4f}")
    return metrics

# FGSM attack
print("Running FGSM attack...")
fgsm = FastGradientMethod(estimator=classifier, eps=epsilon)
x_adv_fgsm = fgsm.generate(x=x_test)
attack_results['FGSM'] = evaluate_attack(classifier, x_adv_fgsm, x_test, y_test, 'FGSM', attack_results)

# PGD-20 attack
print("\nRunning PGD-20 attack...")
pgd = ProjectedGradientDescent(estimator=classifier, eps=epsilon, eps_step=2/255, max_iter=20, random_init=True)
x_adv_pgd = pgd.generate(x=x_test)
attack_results['PGD-20'] = evaluate_attack(classifier, x_adv_pgd, x_test, y_test, 'PGD-20', attack_results)

# C&W L2 attack
print("\nRunning C&W L2 attack...")
cw = CarliniL2Method(estimator=classifier, max_iter=50, confidence=0.0)
x_adv_cw = cw.generate(x=x_test)
attack_results['C&W L2'] = evaluate_attack(classifier, x_adv_cw, x_test, y_test, 'C&W L2', attack_results)

# Create structured audit report
audit_report = pd.DataFrame.from_dict(attack_results, orient='index')
audit_report.columns = ['Fooling Rate (%)', 'Mean L2 Magnitude', 'Mean L∞ Magnitude']
audit_report = audit_report.round(4)

print("\n" + "="*70)
print("VULNERABILITY AUDIT REPORT")
print("="*70)
print(f"Model: SimpleCNN on CIFAR-10")
print(f"Test set size: {len(x_test)} examples")
print(f"Clean accuracy: {clean_accuracy:.1%}")
print("\nPer-Attack Metrics:")
print(audit_report)

# Determine primary defense recommendation
max_fooling_attack = audit_report['Fooling Rate (%)'].idxmax()
max_fooling_rate = audit_report['Fooling Rate (%)'].max()

print("\nDefense Recommendation:")
if max_fooling_rate > 80:
    severity = "CRITICAL"
elif max_fooling_rate > 50:
    severity = "HIGH"
elif max_fooling_rate > 20:
    severity = "MEDIUM"
else:
    severity = "LOW"

print(f"Severity: {severity}")
print(f"Primary attack: {max_fooling_attack} ({max_fooling_rate:.1f}% fooling rate)")
print(f"Recommended action: {'Adversarial training' if max_fooling_rate > 50 else 'Input preprocessing or ensemble defenses'}")
print("="*70)

In [ ]:
# Run AutoAttack
from autoattack import AutoAttack

print("Running AutoAttack...")

adversary = AutoAttack(model, 
                       norm='Linf', 
                       eps=8/255, 
                       version='custom', 
                       attacks_to_run=['apgd-ce', 'apgd-dlr'])
adversary.apgd.n_restarts = 1
x_adv = adversary.run_standard_evaluation(x_test, y_test)

## Expected Outputs and Questions/Tasks

After running the attacks above, you should see:

- FGSM fooling rate between 20% and 60%
- PGD-20 fooling rate between 40% and 80% (higher than FGSM, since PGD is stronger)
- C&W L2 fooling rate between 50% and 90% (often the highest, since C&W is designed to minimize perturbation)
- Mean L2 perturbation magnitudes between 0.01 and 0.30
- Mean L∞ perturbation magnitudes between 0.01 and 0.15

Your audit report displays these values in a structured table format with one row per attack and three metric columns.

1. Why is PGD-20 typically more effective (higher fooling rate) than FGSM, even though both operate within the same epsilon budget?
2. The C&W attack minimizes perturbation norm rather than maximizing loss. Based on your audit report, which attack found the smallest perturbations? What does this tell you about the decision boundary geometry?

In [ ]:
# Verification/Testing Cells
# Verification 1: Check that all three attacks have been run
required_attacks = ['FGSM', 'PGD-20', 'C&W L2']
assert all(attack in audit_report.index for attack in required_attacks), \
    f"Missing attacks. Expected {required_attacks}, got {list(audit_report.index)}"
print("PASS: All three attacks present in audit report.")

# Verification 2: Check that fooling rates are in valid range [0, 100]
assert audit_report['Fooling Rate (%)'].min() >= 0 and audit_report['Fooling Rate (%)'].max() <= 100, \
    f"Fooling rates out of range: {audit_report['Fooling Rate (%)'].tolist()}"
print("PASS: All fooling rates in valid range [0, 100].")

# Verification 3: Check that perturbation magnitudes are positive and reasonable
assert audit_report['Mean L2 Magnitude'].min() > 0, \
    f"L2 magnitudes must be positive: {audit_report['Mean L2 Magnitude'].tolist()}"
assert audit_report['Mean L2 Magnitude'].max() <= 1.0, \
    f"L2 magnitudes too large (>1.0): {audit_report['Mean L2 Magnitude'].tolist()}"
print("PASS: Perturbation magnitudes are valid.")

# Verification 4: Check that report table has correct dimensions
assert audit_report.shape == (3, 3), \
    f"Report shape incorrect: expected (3, 3), got {audit_report.shape}"
print("PASS: Audit report has correct dimensions (3 attacks x 3 metrics).")

## Optional Challenges

1. **RobustBench Leaderboard Simulation**: Retrieve the supplied RobustBench CIFAR-10 leaderboard snapshot (sample data at epsilon 8/255). Identify the closest published AutoAttack accuracy to your model's estimated robustness. Which defense strategy (adversarial training, certified, pre-processing) appears most frequently at the top of the leaderboard?

2. **Continuous Monitoring Setup**: Modify the audit code to run on multiple model checkpoints (e.g., after training epochs 1, 5, 10, 20). Log the per-attack fooling rate for each checkpoint using a simple CSV file or a mock MLflow experiment. Plot the fooling rate trends to identify if robustness degrades or improves as training progresses.

In [ ]:
# Cleanup: Clear GPU memory and remove temporary CIFAR-10 cache if desired
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

# Optional: remove CIFAR-10 download cache
# Uncomment the line below only if you want to remove the downloaded dataset
# import shutil
# shutil.rmtree('/tmp/cifar10', ignore_errors=True)
# print("CIFAR-10 cache removed.")

print("Cleanup complete.")

## Completion Criteria

You have completed this lab when:

- [ ] You have defined a SimpleCNN model and wrapped it in an ART PyTorchClassifier.
- [ ] You have loaded the CIFAR-10 test set (first 200 examples) and computed baseline clean accuracy.
- [ ] You have run three attacks (FGSM, PGD-20, C&W L2) via ART on the test batch.
- [ ] You have produced a structured audit report (pandas DataFrame) with fooling rate and perturbation magnitudes for each attack.
- [ ] You have computed per-attack metrics and verified they are in expected ranges (Verification cells all print PASS).
- [ ] You have answered both comprehension questions about attack effectiveness and decision boundary geometry.
- [ ] You have (optionally) attempted one or both optional challenges to integrate leaderboard lookup or continuous monitoring.